### 1. Load Benchmark Problems

In [1]:
sampling_for_lite = False # True for MineCEraft Lite
# random_seed: single source of truth from builder.json (benchmark sampling + LLM seed when supported)
from pathlib import Path
import json
_builder_path = Path.cwd().parent / "builder.json"
_builder_cfg = json.loads(_builder_path.read_text(encoding="utf-8")) if _builder_path.exists() else {}
sampling_rand_seed = _builder_cfg.get("random_seed", 42)

In [2]:
from pathlib import Path
import json
import random

# Load JSON files from benchmarks folder only (not archive/; archive = excluded from evaluation)
# Schema: prompts = [[turn1, turn2, ...], ...], checks = [[eval_turn1], [eval_turn2], ...]
BENCHMARKS_DIR = Path.cwd() / "benchmarks"
json_files = sorted(BENCHMARKS_DIR.glob("*.json"))
mapping = []
for json_file in json_files:
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

# Build runs: each run = (prompt_sequence, checks_per_turn, _comment)
runs = []
if sampling_for_lite:
    rng = random.Random(sampling_rand_seed)
    for json_file in json_files:
        file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
        file_runs = []
        for item in file_mapping:
            for prompt_sequence in item["prompts"]:
                file_runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))
        if file_runs:
            runs.append(rng.choice(file_runs))
else:
    for item in mapping:
        for prompt_sequence in item["prompts"]:
            runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))

print(f"[PY] Benchmarks dir: {BENCHMARKS_DIR.resolve()} (cwd: {Path.cwd().resolve()})")
if len(runs) == 0:
    print("[PY] ⚠️ No runs. Put at least one .json in benchmarks/ (not in archive/), or run notebook from the folder that contains benchmarks/.")
print("Total problem #:", len(runs))
for i, (prompts, _, _) in enumerate(runs):
    print(i, prompts)

[PY] Benchmarks dir: D:\git\mineCEraft\mineCEraft\benchmarks (cwd: D:\git\mineCEraft\mineCEraft)
Total problem #: 15
0 ['Build a studio.', 'The client changed their mind. Revise it to a two-room house.']
1 ['Build a studio.', 'The client changed their mind. Revise it to a three-room house.']
2 ['Build a studio.', 'The client changed their mind. Revise it to a four-room house.']
3 ['Build a two-room house.', 'The client changed their mind. Revise it to a studio.']
4 ['Build a two-room house.', 'The client changed their mind. Revise it to a three-room house.']
5 ['Build a two-room house.', 'The client changed their mind. Revise it to a four-room house.']
6 ['Build a three-room house.', 'The client changed their mind. Revise it to a studio.']
7 ['Build a three-room house.', 'The client changed their mind. Revise it to a two-room house.']
8 ['Build a three-room house.', 'The client changed their mind. Revise it to a four-room house.']
9 ['Build a four-room house.', 'The client changed thei

### 2. Build phase (construction)

This cell runs the construction (builder agent) only and writes an `eval_*_raw.json` file that contains, for each prompt, its checks and cumulative coordinates. Run the "Load Benchmark Problems" cell above first so that `runs` is defined.

In [3]:
from pathlib import Path
from build_eval_raw import run_build_and_save_eval_raw

# Build phase:
# - uses `runs` constructed in the "Load Benchmark Problems" cell
# - talks to the builder agent
# - writes a single eval_{model_safe}_{ts}_raw.json file under eval_results/

try:
    runs  # type: ignore[name-defined]
except NameError as exc:
    raise RuntimeError("`runs` is not defined. Run the 'Load Benchmark Problems' cell above first.") from exc

eval_raw_path = run_build_and_save_eval_raw(runs)
# For downstream evaluation we treat eval_raw as a list of files.
eval_raw_files = [eval_raw_path]

print(f"[PY] eval_raw file for evaluation: {eval_raw_path}")

[PY] Using builder model: meta-llama/llama-4-scout-17b-16e-instruct (safe='meta-llama-llama-4-scout-17b-16e-instruct')
[PY] Total turns to send: 39
[PY] Intermediate eval_raw file: eval_results\eval_meta-llama-llama-4-scout-17b-16e-instruct_20260316_183018_raw.json
[PY] Builder agent started (PID=60380)
[PY] Agent log: eval_results\eval_meta-llama-llama-4-scout-17b-16e-instruct_20260316_183018_agent.log
ℹ️ Using agent name: builder
🧹 Cleared all files under D:\git\mineCEraft\bots\builder\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=ZOQodMB58Dsqdz4bAAAD)

➡️ Sending to builder (run 1, turn 1/2): "Build a studio."
⏳ Waiting for completion keyword (timeout 10 min)...
📨 [builder]  !newAction("Design and build a studio structure using oak_planks and oak_door, starting from current position.")
📨 [builder] // Get current position const position = world.getPosition(bot);  // Define studio dimensions const width = 10; const depth = 10; const height = 5;  // Build wa

### 3. Evaluation phase (eval_raw → log/csv)

This cell reads one or more `eval_*_raw.json` files and produces `eval_{model_safe}_{ts}.log` and `eval_{model_safe}_{ts}.csv`. It also populates `coords_by_problem` for the visualization cell.

In [4]:
from eval_from_raw import evaluate_from_raw

## Optional: evaluate a custom list of eval_*_raw.json files instead of the default one.
## Example:
# eval_raw_files = [
#     "eval_results/eval_gemini-3-pro-preview_20260311_170420_raw.json",
# ]

# Default: evaluate the eval_raw file produced in the build phase above.
coords_by_problem, log_path, csv_path = evaluate_from_raw(eval_raw_files)

print(f"[PY] Log path: {log_path}")
print(f"[PY] CSV path: {csv_path}")

[PY] Reading eval_raw from eval_results\eval_meta-llama-llama-4-scout-17b-16e-instruct_20260316_183018_raw.json

[PY] === Evaluation Result ===
[PY] Run #1, Turn #1/2: Build a studio.
[PY] Score: 3.0 / 3 (coords=344)
[PY] Category scores:
  - physical_plausibility: 1.0 / 1
  - shape: 2.0 / 2
  · PASS | eval_code.physical_plausibility.is_ground_connected({})
  · PASS | eval_code.shape.has_rooms({'room_cnt': 1})
  · PASS | eval_code.shape.are_doors_passable({})

[PY] === Evaluation Result ===
[PY] Run #1, Turn #2/2: The client changed their mind. Revise it to a two-room house.
[PY] Score: 2.0 / 3 (coords=408)
[PY] Category scores:
  - physical_plausibility: 1.0 / 1
  - shape: 1.0 / 2
  · PASS | eval_code.physical_plausibility.is_ground_connected({})
  · PASS | eval_code.shape.has_rooms({'room_cnt': 2})
  · FAIL | eval_code.shape.are_doors_passable({})

[PY] === Evaluation Result ===
[PY] Run #2, Turn #1/2: Build a studio.
[PY] Score: 2.0 / 3 (coords=308)
[PY] Category scores:
  - physica